# Jet Classification via Graph Neural Networks (Colab Ready)
This notebook implements a **ParticleNet**-style GNN for Quark vs. Gluon jet classification using PyTorch Geometric.

In [1]:
# --- 1. SETUP & IMPORTS ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q h5py torch-geometric
    import torch
    !pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-{torch.__version__}.html

import h5py, os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import EdgeConv, global_mean_pool
from torch_geometric.data import Dataset, Data
from torch_geometric.loader import DataLoader
from torch.utils.data import random_split
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score

# --- CONFIGURATION ---
DATA_PATH = '/content/drive/MyDrive/Research Papers/GSOC_Tasks/quark-gluon_data-set_n139306.hdf5'
GRID_SIZE, WINDOW_SIZE, CENTER_OFFSET = 125, 0.8, 62.0

import seaborn as sns
from sklearn.metrics import roc_curve, auc

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 78.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 135.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 45.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 117.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 73.4 MB/s eta 0:00:00


## 2. Processing & Utilities

In [2]:
def image_to_point_cloud(image, threshold=1e-6):
    active_mask = np.sum(image, axis=-1) > threshold
    active_indices = np.argwhere(active_mask)
    if len(active_indices) == 0: return np.zeros((0, 5))
    row_idx, col_idx = active_indices[:, 0], active_indices[:, 1]
    eta = (row_idx - CENTER_OFFSET) / GRID_SIZE * WINDOW_SIZE
    phi = (col_idx - CENTER_OFFSET) / GRID_SIZE * WINDOW_SIZE
    return np.column_stack([eta, phi, image[row_idx, col_idx, :]])

def build_jet_graph(point_cloud, label, k=16):
    from torch_geometric.nn import knn_graph
    pos = torch.tensor(point_cloud[:, :2], dtype=torch.float)
    x = torch.tensor(point_cloud, dtype=torch.float)
    edge_index = knn_graph(pos, k=k)
    return Data(x=x, edge_index=edge_index, pos=pos, y=torch.tensor([int(label)], dtype=torch.long))

def visualize_jet_graph_3d(data, title="Jet Graph 3D"):
    pos, edge_index = data.pos.numpy(), data.edge_index.numpy()
    z = data.x[:, 2:].sum(dim=1).numpy()
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    for i in range(edge_index.shape[1]):
        ax.plot(pos[edge_index[:, i], 0], pos[edge_index[:, i], 1], z[edge_index[:, i]], color='gray', alpha=0.1)
    ax.scatter(pos[:, 0], pos[:, 1], z, c=z, cmap='viridis', s=20)
    plt.show()

## 3. GNN Architecture & Pipeline

In [3]:
class ParticleNet(nn.Module):
    def __init__(self, input_dim=5, num_classes=2):
        super(ParticleNet, self).__init__()
        # EdgeConv's MLP: Linear -> BatchNorm -> ReLU -> Linear -> BatchNorm -> ReLU
        self.conv1 = EdgeConv(nn.Sequential(
            nn.Linear(2*input_dim, 64),
            nn.BatchNorm1d(64), # Add BatchNorm
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.BatchNorm1d(64), # Add BatchNorm
            nn.ReLU()
        ), aggr='mean')
        self.conv2 = EdgeConv(nn.Sequential(
            nn.Linear(2*64, 128),
            nn.BatchNorm1d(128), # Add BatchNorm
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.BatchNorm1d(128), # Add BatchNorm
            nn.ReLU()
        ), aggr='mean')
        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.BatchNorm1d(64), # BatchNorm after first linear layer in FC
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, num_classes)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, getattr(data, 'batch', None)
        if batch is None: batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)

        x = self.conv1(x, edge_index)
        if torch.isnan(x).any():
            print("NaN detected after conv1")

        x = self.conv2(x, edge_index)
        if torch.isnan(x).any():
            print("NaN detected after conv2")

        x = global_mean_pool(x, batch)
        if torch.isnan(x).any():
            print("NaN detected after global_mean_pool")

        x = self.fc(x)
        if torch.isnan(x).any():
            print("NaN detected after final FC")

        return x

In [4]:
from torch_geometric.data import Dataset

class JetGraphDataset(Dataset):
    def __init__(self, data_path, k=16):
        super().__init__()
        self.k = k
        print(f"Loading data from {data_path} and preprocessing into graphs (this may take a while)...")
        self.graphs = []
        with h5py.File(data_path, 'r') as f:
            X_jets_raw = f['X_jets'][:]
            y_raw = f['y'][:]

        for idx in range(len(y_raw)):
            pts = image_to_point_cloud(X_jets_raw[idx])
            # Only add graph if point cloud is not empty
            if pts.shape[0] > 0:
                graph = build_jet_graph(pts, y_raw[idx], k=self.k)
                self.graphs.append(graph)
        self.num_samples = len(self.graphs)
        print(f"Preprocessing complete. Total valid graphs: {self.num_samples}")

    def len(self):
        return self.num_samples

    def get(self, idx):
        return self.graphs[idx]

def train_epoch(model, loader, optimizer, criterion, device, current_epoch, scaler=None):
    model.train()
    total_loss = 0
    for i, data in enumerate(loader):
        data = data.to(device)
        optimizer.zero_grad()

        # --- Debugging NaN inputs ---
        if torch.isnan(data.x).any():
            print(f"Epoch {current_epoch}, Batch {i}: NaN detected in input data.x.")
            raise ValueError("NaN detected in input data.x")

        # Use autocast for mixed precision if scaler is provided
        with torch.amp.autocast(device_type=device.type, enabled=scaler is not None):
            outputs = model(data)
            loss = criterion(outputs, data.y)

        # --- Debugging NaN outputs ---
        if torch.isnan(outputs).any():
            print(f"Epoch {current_epoch}, Batch {i}: NaN detected in model outputs.")
            raise ValueError("NaN detected in model outputs. Halting training for inspection.")

        if scaler is not None:
            scaler.scale(loss).backward()
            # Unscales the gradients of optimizer's assigned params in-place
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Apply gradient clipping
            optimizer.step()
        total_loss += loss.item() * data.num_graphs
    return total_loss / len(loader.dataset)

def evaluate(model, loader, criterion, device):
    model.eval()
    y_true, y_probs = [], []
    total_loss = 0
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data)
            loss = criterion(out, data.y)
            total_loss += loss.item() * data.num_graphs
            y_true.extend(data.y.cpu().numpy())
            y_probs.extend(torch.softmax(out, dim=1).cpu().numpy())
    y_true, y_probs = np.array(y_true), np.array(y_probs)
    y_preds = np.argmax(y_probs, axis=1)
    auc_score = roc_auc_score(y_true, y_probs[:, 1])
    acc = accuracy_score(y_true, y_preds)
    cm = confusion_matrix(y_true, y_preds)
    return auc_score, acc, cm, y_true, y_probs, total_loss / len(loader.dataset)

def plot_performance(y_true, y_probs, cm, save_path=None):
    plt.figure(figsize=(15, 6))

    # 1. Confusion Matrix
    plt.subplot(1, 2, 1)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Gluon', 'Quark'], yticklabels=['Gluon', 'Quark'])
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')

    # 2. ROC Curve
    fpr, tpr, _ = roc_curve(y_true, y_probs[:, 1])
    roc_auc = auc(fpr, tpr)
    plt.subplot(1, 2, 2)
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC)')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
        print(f"Performance plot saved to {save_path}")
    plt.show()

def plot_training_curves(history, save_path=None):
    plt.figure(figsize=(15, 5))

    # 1. Loss Curve
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Loss')
    plt.legend()
    plt.grid(alpha=0.3)

    # 2. AUC Curve
    plt.subplot(1, 2, 2)
    plt.plot(history['val_auc'], label='Val AUC', color='green')
    plt.xlabel('Epoch')
    plt.ylabel('AUC')
    plt.title('Validation ROC-AUC')
    plt.legend()
    plt.grid(alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path)
        print(f"Training curves plot saved to {save_path}")
    plt.show()

In [5]:
## --- 3. DATA & MODEL INITIALIZATION ---
full_dataset = JetGraphDataset(DATA_PATH, k=16)
train_size = int(0.8 * len(full_dataset))
val_size = int(0.1 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size
train_set, val_set, test_set = random_split(full_dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32)
test_loader = DataLoader(test_set, batch_size=32)

Loading data from /content/drive/MyDrive/Research Papers/GSOC_Tasks/quark-gluon_data-set_n139306.hdf5 and preprocessing into graphs (this may take a while)...
Preprocessing complete. Total valid graphs: 139306


In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ParticleNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001) # Reduced learning rate
criterion = nn.CrossEntropyLoss()

history = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_acc': []}
CHECKPOINT_PATH = '/content/drive/MyDrive/Research Papers/GSOC_Tasks/GNN/latest_checkpoint_new.pth'
BEST_MODEL_PATH = '/content/drive/MyDrive/Research Papers/GSOC_Tasks/GNN/best_model_new.pth'
start_epoch = 1
best_auc = 0.0

if os.path.exists(CHECKPOINT_PATH):
    print(f"Found checkpoint at {CHECKPOINT_PATH}. Resuming training...")
    # Load the checkpoint with weights_only=False to allow custom objects
    checkpoint = torch.load(CHECKPOINT_PATH, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_auc = checkpoint['best_auc']
    history = checkpoint.get('history', history)
    print(f"Resuming from Epoch {start_epoch} (Best AUC so far: {best_auc:.4f})")

Found checkpoint at /content/drive/MyDrive/Research Papers/GSOC_Tasks/GNN/latest_checkpoint_new.pth. Resuming training...
Resuming from Epoch 41 (Best AUC so far: 0.8057)


In [ ]:
## --- 4. TRAINING LOOP ---
num_epochs = 50
# Initialize GradScaler for mixed precision training
scaler = torch.amp.GradScaler() if device.type == 'cuda' else None

# Enable anomaly detection for more detailed NaN traceback
torch.autograd.set_detect_anomaly(True)

for epoch in range(start_epoch, num_epochs + 1):
    loss = train_epoch(model, train_loader, optimizer, criterion, device, epoch, scaler)
    auc_val, acc_val, cm_val, _, _, val_loss = evaluate(model, val_loader, criterion, device)

    history['train_loss'].append(loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(auc_val)
    history['val_acc'].append(acc_val)

    # Save latest checkpoint for resuming
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_auc': max(best_auc, auc_val),
        'history': history
    }, CHECKPOINT_PATH)

    if auc_val > best_auc:
        best_auc = auc_val
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f'Epoch {epoch:02d} | Loss: {loss:.4f} | Val AUC: {auc_val:.4f} (Saved Best!)')
    else:
        print(f'Epoch {epoch:02d} | Loss: {loss:.4f} | Val AUC: {auc_val:.4f}')

## --- 5. FINAL VISUALIZATION ---
plots_dir = '/content/drive/MyDrive/Research Papers/GSOC_Tasks/GNN/plots_new'
os.makedirs(plots_dir, exist_ok=True)

plot_training_curves(history, save_path=os.path.join(plots_dir, 'training_curves.png'))

print("\nLoading best model for final testing...")
if os.path.exists(BEST_MODEL_PATH):
    model.load_state_dict(torch.load(BEST_MODEL_PATH))
    auc_test, acc_test, cm_test, y_true_test, y_probs_test, _ = evaluate(model, test_loader, criterion, device)
    print(f"Final Test Performance | AUC: {auc_test:.4f} | Acc: {acc_test:.4f}")
    plot_performance(y_true_test, y_probs_test, cm_test, save_path=os.path.join(plots_dir, 'test_performance.png'))